In [1]:
import sys
sys.path.append("..")

import time
import joblib
import numpy as np
import pandas as pd
import shap

from src.db import get_engine
from src.policy import load_policy, decide

model = joblib.load("../models/lightgbm_v2_calibrated.joblib")
prep = model.pipeline.named_steps["prep"]
lgbm = model.pipeline.named_steps["model"]
policy = load_policy("../results/policy.json")

test = pd.read_parquet("../data/processed/test.parquet")
predictions = pd.read_parquet("../data/processed/test_predictions.parquet")

print("Same order:", (test["id"].values == predictions["id"].values).all())

c:\Users\Administrator\loan-default-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Same order: True


In [2]:
start = time.time()

X_ready = prep.transform(test.drop(columns=["target"]))
explainer = shap.TreeExplainer(lgbm)
values = explainer.shap_values(X_ready)

if isinstance(values, list):
    values = values[1]
if values.ndim == 3:
    values = values[:, :, 1]

print(values.shape, f"in {time.time() - start:.0f} seconds")

(162236, 62) in 57 seconds


c:\Users\Administrator\loan-default-project\.venv\Lib\site-packages\shap\explainers\_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [3]:
feature_names = np.array(X_ready.columns)

top3 = np.argsort(-np.abs(values), axis=1)[:, :3]
rows = np.arange(len(values))[:, None]
top3_contributions = values[rows, top3]

table = pd.DataFrame({
    "id": predictions["id"].values,
    "split": "test",
    "model_name": "lightgbm_v2_calibrated",
    "probability": predictions["p_lgbm"].values,
    "actual_default": predictions["target"].values,
})
table["decision"] = table["probability"].apply(
    lambda p: decide(p, policy["cutoff"], policy["review_margin"])
)

for k in range(3):
    table[f"reason_{k + 1}"] = feature_names[top3[:, k]]
    table[f"reason_{k + 1}_contribution"] = top3_contributions[:, k]
    table[f"reason_{k + 1}_effect"] = np.where(
        top3_contributions[:, k] > 0, "raises risk", "lowers risk"
    )

table.head()

,id,split,model_name,probability,actual_default,decision,reason_1,reason_1_contribution,reason_1_effect,reason_2,reason_2_contribution,reason_2_effect,reason_3,reason_3_contribution,reason_3_effect
0,56705,test,lightgbm_v2_calibrated,0.131302,0,Approve,loan_to_income,0.240413,raises risk,annual_inc,0.213757,raises risk,inq_last_6mths,-0.079575,lowers risk
1,67025,test,lightgbm_v2_calibrated,0.036785,0,Approve,fico_avg,-0.458221,lowers risk,purpose,-0.176655,lowers risk,acc_open_past_24mths,-0.129095,lowers risk
2,361774,test,lightgbm_v2_calibrated,0.068592,0,Approve,loan_to_income,-0.220941,lowers risk,credit_history_months,0.190617,raises risk,annual_inc,-0.188265,lowers risk
3,365090,test,lightgbm_v2_calibrated,0.322731,0,Approve,fico_avg,0.320524,raises risk,dti,0.192232,raises risk,addr_state,0.153008,raises risk
4,369183,test,lightgbm_v2_calibrated,0.126544,0,Approve,purpose,0.455534,raises risk,fico_avg,0.177050,raises risk,acc_open_past_24mths,-0.135484,lowers risk


In [4]:
engine = get_engine(fast_executemany=True)

start = time.time()
table.to_sql("predictions", engine, schema="dbo",
             if_exists="append", index=False, chunksize=10_000)
print(f"Inserted {len(table):,} rows in {time.time() - start:.0f} seconds")

Inserted 162,236 rows in 24 seconds
